<a href="https://colab.research.google.com/github/Dharshini1701/priyadharshini/blob/main/langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langchain langchain-google-genai


In [ ]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [26]:
import sqlite3

conn = sqlite3.connect("sales_nl2sql.db")

print("Database connected!")

Database connected!


In [33]:
import sqlite3

conn = sqlite3.connect("sales_nl2sql.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS sales (
    SaleID INTEGER PRIMARY KEY,
    ProductName TEXT,
    Category TEXT,
    City TEXT,
    Quantity INTEGER,
    Amount REAL,
    SaleDate TEXT,
    Status TEXT
)
""")


In [34]:
data = [
    (1, "Laptop", "Electronics", "Chennai", 2, 120000, "2026-01-10", "Completed"),
    (2, "Mobile", "Electronics", "Coimbatore", 3, 75000, "2026-01-12", "Completed"),
    (3, "Headphones", "Accessories", "Bengaluru", 5, 25000, "2026-01-15", "Completed"),
    (4, "Tablet", "Electronics", "Chennai", 2, 60000, "2026-01-20", "Cancelled"),
    (5, "Keyboard", "Accessories", "Madurai", 4, 12000, "2026-01-22", "Completed"),
    (6, "Monitor", "Electronics", "Coimbatore", 3, 45000, "2026-02-02", "Completed"),
    (7, "Mouse", "Accessories", "Chennai", 10, 10000, "2026-02-05", "Completed"),
    (8, "Printer", "Electronics", "Bengaluru", 2, 30000, "2026-02-10", "Completed"),
    (9, "Laptop", "Electronics", "Madurai", 1, 65000, "2026-02-15", "Completed"),
    (10, "Mobile", "Electronics", "Chennai", 4, 100000, "2026-02-20", "Completed")
]

cursor.executemany("""
INSERT INTO sales
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", data)

conn.commit()

print("Data inserted successfully!")



Data inserted successfully!


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.environ["GEMINI_API_KEY"],

)

print("Gemini connected successfully!")

Gemini connected successfully!


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple words."
)

print("Prompt created successfully!")

Prompt created successfully!


In [ ]:
messages = prompt.invoke({
    "topic": "Python"
})

print(messages)

messages=[HumanMessage(content='Explain Python in simple words.', additional_kwargs={}, response_metadata={})]


In [ ]:
response = llm.invoke(messages)

print(response.content)

[{'type': 'text', 'text': '**Python** is a popular computer programming language. \n\nThink of a programming language as a **translator**. Computers only understand ones and zeros, which is impossible for most humans to read. Python acts as a bridge, allowing you to give instructions to a computer using words and numbers that make sense to humans.\n\nHere is why Python is so famous, explained simply:\n\n---\n\n### 1. It’s written in plain English\nMany programming languages look like scary, complicated math equations. Python was designed to be clean and simple. Reading Python code often feels like reading basic English sentences. \n\n* *For example, to make a computer say "Hello!", in Python you literally just type:* `print("Hello!")`\n\n### 2. It’s a "Swiss Army Knife"\nPython is extremely versatile. You can use it to build almost anything, such as:\n* **Artificial Intelligence & Robots:** Tools like ChatGPT and self-driving cars rely heavily on Python.\n* **Websites:** Popular sites 

In [29]:
schema = """
TABLE: sales
COLUMNS:
SaleID INTEGER
ProductName TEXT
Category TEXT
City TEXT
Quantity INTEGER
Amount REAL
sales text
SaleDate TEXT
Status TEXT
"""

print(schema)


TABLE: sales
COLUMNS:
SaleID INTEGER
ProductName TEXT
Category TEXT
City TEXT
Quantity INTEGER
Amount REAL
sales text
SaleDate TEXT
Status TEXT



In [ ]:
from langchain_core.prompts import ChatPromptTemplate

sql_prompt = ChatPromptTemplate.from_template(
    """
You are an expert SQLite SQL generator.

DATABASE SCHEMA:
{schema}

Convert the user's question into a SQLite SELECT query.

Rules:
- Only SELECT
- Only use the sales table
- Only use columns from the schema
- Return only SQL
- No explanation
- No markdown

USER QUESTION:
{question}
"""
)



In [ ]:
question = "What is the total sales in Chennai?"

messages = sql_prompt.invoke({
    "schema": schema,
    "question": question
})

print(messages)


messages=[HumanMessage(content="\nYou are an expert SQLite SQL generator.\n\nDATABASE SCHEMA:\n\nTABLE: sales\nCOLUMNS:\nSaleID INTEGER\nProductName TEXT\nCategory TEXT\nCity TEXT\nQuantity INTEGER\nAmount REAL\nSaleDate TEXT\nStatus TEXT\n\n\nConvert the user's question into a SQLite SELECT query.\n\nRules:\n- Only SELECT\n- Only use the sales table\n- Only use columns from the schema\n- Return only SQL\n- No explanation\n- No markdown\n\nUSER QUESTION:\nWhat is the total sales in Chennai?\n", additional_kwargs={}, response_metadata={})]


In [ ]:
response = llm.invoke(messages)

print(response.content)

[{'type': 'text', 'text': "SELECT SUM(Amount) FROM sales WHERE City = 'Chennai';", 'extras': {'signature': 'Er0LCroLARFNMg+W00h/WMkkdVBsCVKj3gb+Ao7hssM3c/vEwpW8fQbUW+6JByWq77d05y1WQSliJTkhIM4WW17dbe+mBedQVvNti/txOtuNQ2p+tWhjhprvnt50car9jYCRC5f5qh6M+aIIS5OQGJMBPttowP4vEBNwX/utPYNeEKafC5JlOBg3cai++uuRvXxefVFzeVOI/quL7JRNwk+H7K+HYX1gP5iBVyO/eY6CZIGc0lRpMu/5lixSIqRuXSc9esM1HhW7l3PGvcqLisdZqk1JSOdhQR6vpyRFxyMTBEgMt8hqB4nLy9WLRo1lPkgp8dqKhAQ3BIz0G1oRkv7IFOfbh+JIeYIkJYXiERtxd8vlum/aRYuAB9DS1hgHJlOSxSNZZ4nq3FLdUO6CQQdKjHCjBA4DfEsRLA8fPwBB6f/iU+Odlq5oItdiztgUQJ6HRapK5AryGGyMzYr8Y0qf+zvj0wW/c8vU4CG1EHqR3xEocJoA5em1PRlF9HyZGN3TkzUrEWyNFW/wu3nCroayaGOl6fh5I3kJUzy/+hOLIjdQhciDUZVVFh6eX3g+gr2KNAYB81TD1lGAAO1iAwR/MnEga9c8fr3VOYWYO3aQF+fJeLc1530BSIAXxcBdmxc1i/BOtFR5HGAcujANe/RWTLBhKcDBTSuxHKtQgj215aHsrLWdqMSRXgj7vYGPmMCisVS6ZNnpFevlHwEUUGN4HJyLKjXItrBSUwOHmEgYoStQ/5NBN8nfVtn3aQlbPt2qGYZt3DRE0BCELrmKdfA9SSg2lqQG2n227nnk8cVbPiJ6uSYTj8x4Q/1YIZ8ZIa4BwxnuHhA4wpEI+KsojlUMLd2IvaVWNokTaYz6aZJieVjuhjn5P84E0Lu7

In [ ]:
sql = response.content

print("Generated SQL:")
print(sql)

Generated SQL:
[{'type': 'text', 'text': "SELECT SUM(Amount) FROM sales WHERE City = 'Chennai';", 'extras': {'signature': 'Er0LCroLARFNMg+W00h/WMkkdVBsCVKj3gb+Ao7hssM3c/vEwpW8fQbUW+6JByWq77d05y1WQSliJTkhIM4WW17dbe+mBedQVvNti/txOtuNQ2p+tWhjhprvnt50car9jYCRC5f5qh6M+aIIS5OQGJMBPttowP4vEBNwX/utPYNeEKafC5JlOBg3cai++uuRvXxefVFzeVOI/quL7JRNwk+H7K+HYX1gP5iBVyO/eY6CZIGc0lRpMu/5lixSIqRuXSc9esM1HhW7l3PGvcqLisdZqk1JSOdhQR6vpyRFxyMTBEgMt8hqB4nLy9WLRo1lPkgp8dqKhAQ3BIz0G1oRkv7IFOfbh+JIeYIkJYXiERtxd8vlum/aRYuAB9DS1hgHJlOSxSNZZ4nq3FLdUO6CQQdKjHCjBA4DfEsRLA8fPwBB6f/iU+Odlq5oItdiztgUQJ6HRapK5AryGGyMzYr8Y0qf+zvj0wW/c8vU4CG1EHqR3xEocJoA5em1PRlF9HyZGN3TkzUrEWyNFW/wu3nCroayaGOl6fh5I3kJUzy/+hOLIjdQhciDUZVVFh6eX3g+gr2KNAYB81TD1lGAAO1iAwR/MnEga9c8fr3VOYWYO3aQF+fJeLc1530BSIAXxcBdmxc1i/BOtFR5HGAcujANe/RWTLBhKcDBTSuxHKtQgj215aHsrLWdqMSRXgj7vYGPmMCisVS6ZNnpFevlHwEUUGN4HJyLKjXItrBSUwOHmEgYoStQ/5NBN8nfVtn3aQlbPt2qGYZt3DRE0BCELrmKdfA9SSg2lqQG2n227nnk8cVbPiJ6uSYTj8x4Q/1YIZ8ZIa4BwxnuHhA4wpEI+KsojlUMLd2IvaVWNokTaYz6aZJie

In [ ]:
if isinstance(sql, list):
    sql = sql[0]["text"]

sql = sql.replace("```sql", "").replace("```", "").strip()

print(sql)

SELECT SUM(Amount) FROM sales WHERE City = 'Chennai';


In [35]:
cursor = conn.cursor()

cursor.execute(sql)

result = cursor.fetchone()

print("Result:", result)

Result: (290000.0,)


In [37]:
question = input("Enter your question: ")

print("Your question:", question)


Enter your question: Enter your question: What is the total sales in Madurai?
Your question: Enter your question: What is the total sales in Madurai?


In [38]:
messages = sql_prompt.invoke({
    "schema": schema,
    "question": question
})

print(messages)

messages=[HumanMessage(content="\nYou are an expert SQLite SQL generator.\n\nDATABASE SCHEMA:\n\nTABLE: sales\nCOLUMNS:\nSaleID INTEGER\nProductName TEXT\nCategory TEXT\nCity TEXT\nQuantity INTEGER\nAmount REAL\nsales text\nSaleDate TEXT\nStatus TEXT\n\n\nConvert the user's question into a SQLite SELECT query.\n\nRules:\n- Only SELECT\n- Only use the sales table\n- Only use columns from the schema\n- Return only SQL\n- No explanation\n- No markdown\n\nUSER QUESTION:\nEnter your question: What is the total sales in Madurai?\n", additional_kwargs={}, response_metadata={})]


In [39]:
response = llm.invoke(messages)

print(response.text)

SELECT SUM(Amount) FROM sales WHERE City = 'Madurai'


In [40]:
sql = response.text

print("Generated SQL:")
print(sql)

Generated SQL:
SELECT SUM(Amount) FROM sales WHERE City = 'Madurai'


In [41]:
cursor.execute(sql)

result = cursor.fetchone()

print("Result:", result)

Result: (77000.0,)


In [42]:
print("Question:", question)
print("SQL:", sql)
print("Answer:", result[0])

Question: Enter your question: What is the total sales in Madurai?
SQL: SELECT SUM(Amount) FROM sales WHERE City = 'Madurai'
Answer: 77000.0


In [43]:
def clean_sql(sql):
    return sql.replace("```sql", "").replace("```", "").strip()

sql = clean_sql(response.text)

print(sql)

SELECT SUM(Amount) FROM sales WHERE City = 'Madurai'


In [44]:
def validate_sql(sql):
    sql_upper = sql.upper().strip()

    if not sql_upper.startswith("SELECT"):
        raise ValueError("Only SELECT queries are allowed")

    return sql

sql = validate_sql(sql)

print("SQL is valid!")
print(sql)

SQL is valid!
SELECT SUM(Amount) FROM sales WHERE City = 'Madurai'


In [45]:
cursor.execute(sql)

result = cursor.fetchone()

print("Answer:", result[0])

Answer: 77000.0
